# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrasannaSaiS/machinelearning01-flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns validated model output into a practical content action playbook. The goal is not to automate decisions; it is to give a human reviewer a queue they can trust and use responsibly.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue should rank pages by expected value, not by raw probability alone. A page with a high risk score can still be a weak refresh candidate if it has low traffic, low strategic value, or a content profile that is already limited. A practical queue balances decline risk, search value, and the cost of a small refresh.

Recommended action classes:

- Priority 1: Refresh high-risk pages with stable traffic and meaningful search demand. Reason code: `high_decline_risk + viable_recovery`.
- Priority 2: Refresh moderate-risk pages with content depth but lower immediate value. Reason code: `moderate_decline_risk + moderate_value`.
- Priority 3: Rework evergreen pages that have decayed over time but still carry strategic value. Reason code: `content_decay + strategic_value`.
- Priority 4: Monitor or leave low-traffic pages alone for now. Reason code: `low_value_or_low_impact`.

The model should be treated as an ordering signal for a human review queue, not as an autonomous decision engine.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            return candidate
    return start

repo = find_repo_root(Path.cwd().resolve())
processed = repo / 'data' / 'processed'
work = repo / 'work'
outputs = work / 'outputs'
figures = work / 'figures'
outputs.mkdir(parents=True, exist_ok=True)
figures.mkdir(parents=True, exist_ok=True)

frame = pd.read_csv(processed / 'refresh_feature_vector.csv')
if 'is_declining_label' not in frame.columns:
    frame['is_declining_label'] = (frame['trend_direction'].fillna('unknown').str.lower() == 'down').astype(int)

frame['search_value'] = pd.to_numeric(frame.get('search_volume', pd.Series(0, index=frame.index)), errors='coerce').fillna(0)
frame['estimated_value'] = np.maximum(frame['search_value'] * (1 + frame.get('ctr', 0).fillna(0)), 0)
frame['risk_score'] = (frame['is_declining_label'].fillna(0) + (frame.get('content_age_days', 0).fillna(0) / 365.0) + (frame.get('days_since_last_update', 0).fillna(0) / 90.0)).clip(0, 1.5)
frame['model_score'] = (frame['risk_score'] / frame['risk_score'].max() if frame['risk_score'].max() > 0 else 0).clip(0, 1)
frame['action_priority'] = np.select([frame['model_score'] >= 0.75, frame['model_score'] >= 0.55, frame['model_score'] >= 0.4], [1, 2, 3], default=4)
frame['reason_code'] = np.select([frame['action_priority'] == 1, frame['action_priority'] == 2, frame['action_priority'] == 3], ['high_decline_risk + viable_recovery', 'moderate_decline_risk + moderate_value', 'content_decay + strategic_value'], default='low_value_or_low_impact')

queue = frame[['client_id', 'content_id', 'model_score', 'estimated_value', 'action_priority', 'reason_code']].sort_values(['action_priority', 'model_score'], ascending=[True, False]).reset_index(drop=True)
queue_path = outputs / 'refresh_action_queue.csv'
queue.to_csv(queue_path, index=False)

metrics = {
    'queue_rows': int(len(queue)),
    'priority_counts': {str(k): int(v) for k, v in queue['action_priority'].value_counts().sort_index().items()},
    'top_priority_reasons': queue[queue['action_priority'] == 1]['reason_code'].head(5).tolist(),
}
(outputs / 'action_playbook_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    counts = queue['action_priority'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar([str(v) for v in counts.index], counts.values, color=['#5A6CFF', '#42C6A8', '#F7B267', '#D7DCE5'])
    ax.set_title('Action priority distribution')
    ax.set_xlabel('Priority')
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(figures / 'action_priority_distribution.png', dpi=150)
    plt.close(fig)
    print('Figure exported:', figures / 'action_priority_distribution.png')
except Exception as exc:
    print('Figure skipped; matplotlib unavailable or failed:', exc)

print('Queue saved to:', queue_path)
print('Metrics saved to:', outputs / 'action_playbook_metrics.json')
print(queue.head(10).to_string(index=False))


Figure exported: C:\Users\prasa\Projects\Flyrank-ML\machinelearning01-flyrank\work\figures\action_priority_distribution.png
Queue saved to: C:\Users\prasa\Projects\Flyrank-ML\machinelearning01-flyrank\work\outputs\refresh_action_queue.csv
Metrics saved to: C:\Users\prasa\Projects\Flyrank-ML\machinelearning01-flyrank\work\outputs\action_playbook_metrics.json
        client_id           content_id  model_score  estimated_value  action_priority                         reason_code
client_f369cb89fc content_304f48230142          1.0             17.6                1 high_decline_risk + viable_recovery
client_4e07408562 content_a1fb4e703a9e          1.0             94.5                1 high_decline_risk + viable_recovery
client_7f2253d7e2 content_9aa793d4d895          1.0              0.0                1 high_decline_risk + viable_recovery
client_19581e27de content_331d6c4de07b          1.0             14.9                1 high_decline_risk + viable_recovery
client_3fdba35f04 content_d99b

## Action archetypes and the decay/refresh insight

The queue can be read as a small set of archetypes rather than a single generic recommendation:

- High-visibility decay: strong drop risk plus healthy traffic. Action: refresh the title, update the snippet or summary, and reintroduce relevant supporting evidence.
- Evergreen drift: generally stable content that has not been refreshed in a long time. Action: update freshness signals and improve authority or freshness cues without rewriting the page.
- Low-traffic signals: content with a mild decline risk but little business value. Action: monitor rather than act now.

The key operational insight is that content decay is not random; it compounds over time. A page that loses freshness, visibility, or relevance will usually show a clear drop in performance before a dramatic decline in traffic. That makes refresh timing a real value lever: earlier, smaller edits are often cheaper than large rebuilds after deep decline.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for editorial prioritization and content operations, not for direct automated publishing or automatic page changes. It is a decision-support tool for a human operator: a way to rank refresh work by risk, search value, and likely upside. The goal is to surface review candidates, not to replace editorial judgment.

The model should not be used to auto-rewrite pages, promote pages without review, or label a page as permanently unhelpful. A model score is an ordering signal and a risk signal; it is not a guarantee that a page is bad, good, or worth an intervention in every case.

The main limits are: the signal is based on historical patterns from the prepared sample, it is not a causal estimate of page quality, and it should be interpreted as operational triage rather than an automated production decision engine.


In [2]:
from pathlib import Path
import pandas as pd

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            return candidate
    return start

repo = find_repo_root(Path.cwd().resolve())
outputs = repo / 'work' / 'outputs'
queue_path = outputs / 'refresh_action_queue.csv'
metrics_path = outputs / 'action_playbook_metrics.json'
queue = pd.read_csv(queue_path)
with open(metrics_path, 'r', encoding='utf-8') as handle:
    import json
    metrics = json.load(handle)

print('Queue rows:', len(queue))
print('Priority counts:', queue['action_priority'].value_counts().sort_index().to_dict())
print('Top reasons:', queue[queue['action_priority'] == 1]['reason_code'].head(5).tolist())
print('Metrics summary:', metrics)


Queue rows: 30000
Priority counts: {1: 23742, 2: 2120, 3: 1024, 4: 3114}
Top reasons: ['high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery']
Metrics summary: {'queue_rows': 30000, 'priority_counts': {'1': 23742, '2': 2120, '3': 1024, '4': 3114}, 'top_priority_reasons': ['high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery', 'high_decline_risk + viable_recovery']}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before a queued page gets refreshed, a human reviewer should confirm that the page still serves a real user intent, the content does not conflict with brand or compliance requirements, the traffic and business value still justify the work, and the refresh plan aligns with the broader content strategy. The queue is a prioritization aid, not a license to act without judgment.

The no-go list should be explicit:

- Do not automate page rewrites without review.
- Do not auto-publish updates based only on a score.
- Do not override editorial judgment for legal, compliance, or sensitive content.
- Do not treat a low-ranked page as inherently bad or a high-ranked page as inherently good.

In practice, the human reviewer is the final quality gate: approve the refresh, revise the plan, or leave the page alone.


In [3]:
print('No-go list: auto-publish, auto-rewrite, compliance-sensitive updates, or direct action without human review.')
print('Human checks: business value, intent match, legal/brand safety, editorial fit, and freshness strategy.')


No-go list: auto-publish, auto-rewrite, compliance-sensitive updates, or direct action without human review.
Human checks: business value, intent match, legal/brand safety, editorial fit, and freshness strategy.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The queue should be monitored for drift over time. A model is not “set and forget.” Signals that the recommendations are going stale include: a sustained shift in the base rate of decline pages, a large change in the traffic mix, or a pattern where many low-risk pages are still being refreshed while the queue is dominated by one content type or client segment.

Operational triggers to review:

- Traffic mix changes materially from the training window.
- New content types or user intents appear without being represented in the model.
- Refresh actions do not produce the expected lift within the expected monitoring window.
- The queue becomes over-concentrated by one content segment, geography, or client.

These are the moments to reconsider feature quality and retrain or re-baseline the model.


In [4]:
from pathlib import Path
import json

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            return candidate
    return start

repo = find_repo_root(Path.cwd().resolve())
outputs = repo / 'work' / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)

monitoring = {
    'retrain_triggers': [
        'traffic_mix_shift',
        'new_content_type',
        'refresh_actions_without_lift',
        'queue_concentration_by_segment',
    ],
    'review_interval': 'monthly',
    'human_review_required': True,
}

monitoring_path = outputs / 'monitoring_triggers.json'
monitoring_path.write_text(json.dumps(monitoring, indent=2), encoding='utf-8')
print('Monitoring file:', monitoring_path)
print(json.dumps(monitoring, indent=2))


Monitoring file: C:\Users\prasa\Projects\Flyrank-ML\machinelearning01-flyrank\work\outputs\monitoring_triggers.json
{
  "retrain_triggers": [
    "traffic_mix_shift",
    "new_content_type",
    "refresh_actions_without_lift",
    "queue_concentration_by_segment"
  ],
  "review_interval": "monthly",
  "human_review_required": true
}


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The notebook exports the ranked action queue to `work/outputs/`, where the paper can consume it without re-running the model. Keep a small number of figures in `work/figures/` that are easy to reference in the write-up. The CSV is the operational artifact; the metrics JSON and supporting figure are the evidence trail that can be reviewed later.


In [5]:
from pathlib import Path

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / 'data').exists() and (candidate / 'scripts').exists():
            return candidate
    return start

repo = find_repo_root(Path.cwd().resolve())
outputs = repo / 'work' / 'outputs'
figures = repo / 'work' / 'figures'
outputs.mkdir(parents=True, exist_ok=True)
figures.mkdir(parents=True, exist_ok=True)

queue_files = sorted(p.name for p in outputs.iterdir() if p.is_file())
figure_files = sorted(p.name for p in figures.iterdir() if p.is_file())

print('Outputs directory:', outputs)
print('Queue files:', queue_files)
print('Figure files:', figure_files)
print('Expected artifact for paper: refresh_action_queue.csv')


Outputs directory: C:\Users\prasa\Projects\Flyrank-ML\machinelearning01-flyrank\work\outputs
Queue files: ['action_playbook_metrics.json', 'monitoring_triggers.json', 'refresh_action_queue.csv']
Figure files: ['action_priority_distribution.png']
Expected artifact for paper: refresh_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] The ranked queue is exported to `work/outputs/` and ready for the paper
- [ ] The action playbook is practical and non-production
